In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q08/q08_rewrite/checkpoints/pre_cell_7.pickle

In [4]:
%%RecordEvent
%%time
### cell 7 ###

# Optimized for cudf: eliminate UDF and CPU apply by computing aggregates on GPU
# Filter PART and select key
part_filtered = part[part.P_TYPE == "ECONOMY ANODIZED STEEL"][["P_PARTKEY"]]

# Prepare LINEITEM with volume
lineitem_filtered = lineitem[["L_PARTKEY", "L_SUPPKEY", "L_ORDERKEY", "L_EXTENDEDPRICE", "L_DISCOUNT"]]
lineitem_filtered["VOLUME"] = lineitem_filtered.L_EXTENDEDPRICE * (1.0 - lineitem_filtered.L_DISCOUNT)
lineitem_filtered = lineitem_filtered[["L_PARTKEY", "L_SUPPKEY", "L_ORDERKEY", "VOLUME"]]

# Join PART and LINEITEM
total = part_filtered.merge(
    lineitem_filtered,
    left_on="P_PARTKEY",
    right_on="L_PARTKEY",
    how="inner"
)[["L_SUPPKEY", "L_ORDERKEY", "VOLUME"]]

# Join SUPPLIER to get supplier nation
total = total.merge(
    supplier[["S_SUPPKEY", "S_NATIONKEY"]],
    left_on="L_SUPPKEY",
    right_on="S_SUPPKEY",
    how="inner"
)[["L_ORDERKEY", "VOLUME", "S_NATIONKEY"]]

# Filter ORDERS by date and extract year
orders_filtered = orders[
    (orders.O_ORDERDATE >= "1995-01-01") & (orders.O_ORDERDATE < "1997-01-01")
]
orders_filtered["O_YEAR"] = orders_filtered.O_ORDERDATE.dt.year
orders_filtered = orders_filtered[["O_ORDERKEY", "O_CUSTKEY", "O_YEAR"]]

# Join ORDERS
total = total.merge(
    orders_filtered,
    left_on="L_ORDERKEY",
    right_on="O_ORDERKEY",
    how="inner"
)[["VOLUME", "S_NATIONKEY", "O_CUSTKEY", "O_YEAR"]]

# Join CUSTOMER to get customer nation
total = total.merge(
    customer[["C_CUSTKEY", "C_NATIONKEY"]],
    left_on="O_CUSTKEY",
    right_on="C_CUSTKEY",
    how="inner"
)[["VOLUME", "S_NATIONKEY", "O_YEAR", "C_NATIONKEY"]]

# Join NATION for customer region
total = total.merge(
    nation[["N_NATIONKEY", "N_REGIONKEY"]],
    left_on="C_NATIONKEY",
    right_on="N_NATIONKEY",
    how="inner"
)[["VOLUME", "S_NATIONKEY", "O_YEAR", "N_REGIONKEY"]]

# Join NATION to get supplier nation name
n2 = nation[["N_NATIONKEY", "N_NAME"]].rename(columns={"N_NAME": "NATION"})
total = total.merge(
    n2,
    left_on="S_NATIONKEY",
    right_on="N_NATIONKEY",
    how="inner"
)[["VOLUME", "O_YEAR", "N_REGIONKEY", "NATION"]]

# Filter REGION = AMERICA
region_filtered = region[region.R_NAME == "AMERICA"][["R_REGIONKEY"]]
total = total.merge(
    region_filtered,
    left_on="N_REGIONKEY",
    right_on="R_REGIONKEY",
    how="inner"
)[["VOLUME", "O_YEAR", "NATION"]]

# Compute market share: Brazil volume / total volume per year
# Create a column that is volume for Brazil only
total["VOLUME_BRAZIL"] = total.VOLUME * (total.NATION == "BRAZIL")
# Aggregate sums on GPU
agg = total.groupby("O_YEAR").agg({
    "VOLUME": "sum",
    "VOLUME_BRAZIL": "sum"
})
# Compute share and prepare final result
agg["MKT_SHARE"] = agg.VOLUME_BRAZIL / agg.VOLUME
total = agg["MKT_SHARE"].reset_index().sort_values("O_YEAR")

CPU times: user 95.7 ms, sys: 56.1 ms, total: 152 ms
Wall time: 164 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q08/rewritten/o4_mini_high_small/checkpoints/post_cell_7_try_0.pickle